# Classical Baseline — WENO5 + TVD-RK3

Solves the same four benchmarks with a fifth-order WENO reconstruction of the
convective derivatives and third-order TVD Runge-Kutta time stepping, on the
same 201^3 grid and scored with the same metrics as the PINN.

**Why this is here.** A referee will ask why a neural solver is used for linear
advection when high-order classical schemes are mature and fast. The honest
answer requires the number, not an assertion.

**What is matched.** Grid (201^3), snapshot times, error definitions, and the
reference solution. The WENO solver receives exact Dirichlet data on a
three-cell boundary shell where the exact field is known; the reversed vortex
needs none, since that velocity field vanishes on the faces of the cube.

**No reinitialisation** is applied, so the comparison is like for like: the PINN
maintains the signed-distance property through its eikonal loss term rather
than by periodically re-solving for it.

**Runtime** is CFL-limited and varies by benchmark: roughly 250 steps for
translation, 3000 for rotation and Zalesak, 4000 for the vortex. Expect
5-20 minutes each on a T4.

**Output:** `weno_results.csv`, one row per benchmark.

In [ ]:
# --- setup -------------------------------------------------------------
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive'
except ImportError:
    ROOT = os.getcwd()

# the benchmark cores supply the velocity fields, initial conditions and
# exact solutions; nothing is redefined here
for sub in ['TR3D', 'RO3D', 'ZD3D', 'RV3D', 'WENO']:
    p = f'{ROOT}/{sub}'
    if os.path.isdir(p):
        sys.path.insert(0, p)
sys.path.insert(0, ROOT)

import csv, time
import numpy as np
import torch
import weno_core as weno

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
# --- configuration -----------------------------------------------------
GRID_N      = 201
CFL         = 0.4
N_SNAPSHOTS = 6
RK4_STEPS   = 200                     # RV3D reference back-tracing
OUTDIR      = ROOT
CSV_PATH    = f'{OUTDIR}/weno_results.csv'
os.makedirs(OUTDIR, exist_ok=True)

RUN = ['TR3D', 'RO3D', 'ZD3D', 'RV3D']     # trim this to run one at a time

In [ ]:
# --- per-benchmark setup -----------------------------------------------
def setup(name):
    """Returns everything the solver needs, taken from the benchmark core."""
    if name == 'TR3D':
        import tr3d_core as core
        T = 5.0
    elif name == 'RO3D':
        import ro3d_core as core
        T = 2 * np.pi
    elif name == 'ZD3D':
        import zd3d_core as core
        T = 2 * np.pi
    elif name == 'RV3D':
        import rv3d_core as core
        T = 2.0
    else:
        raise ValueError(name)

    grid = core.Grid(n=GRID_N)
    times, _ = core.snapshot_times(T, N_SNAPSHOTS)
    tt = lambda a: torch.as_tensor(a, dtype=torch.float32, device=device)
    X, Y, Z = tt(grid.X), tt(grid.Y), tt(grid.Z)

    if name == 'RV3D':
        # velocity depends on normalised time; it vanishes on the boundary, so
        # no Dirichlet data is required
        def vel(t):
            tau = torch.full_like(X, t / T)
            return core.velocity_field(X, Y, Z, tau)
        bc_fn = None
        ref_cache = f'{ROOT}/RV3D/RV3D_reference_n{GRID_N}_T{T:g}.npz'
        if not os.path.isdir(os.path.dirname(ref_cache)):
            ref_cache = f'{OUTDIR}/RV3D_reference_n{GRID_N}_T{T:g}.npz'
        PHI_E = core.reference_fields(grid, times, T, device, nsteps=RK4_STEPS,
                                      cache_path=ref_cache)
        phi0 = PHI_E[times[0]]
        V_exact = core.V_EXACT
    else:
        U, V, W = core.velocity_field(X, Y, Z)      # steady for these three
        vel = lambda t: (U, V, W)
        bc_fn = lambda t, Xs, Ys, Zs: core.exact_np(Xs, Ys, Zs, t)
        PHI_E = {t: core.exact_np(grid.X, grid.Y, grid.Z, t) for t in times}
        phi0 = PHI_E[times[0]]
        V_exact = (core.V_EXACT if name != 'ZD3D'
                   else float(np.sum(PHI_E[times[0]] < 0) * grid.d**3))

    return core, grid, times, T, phi0, vel, bc_fn, PHI_E, V_exact

In [ ]:
# --- run ---------------------------------------------------------------
rows = []
for name in RUN:
    print(f'\n=== {name} ===', flush=True)
    core, grid, times, T, phi0, vel, bc_fn, PHI_E, V_exact = setup(name)

    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    sol = weno.solve(phi0, vel, T, grid.d, times, cfl=CFL, device=device,
                     bc_fn=bc_fn,
                     bc_grid=(grid.X, grid.Y, grid.Z) if bc_fn else None)

    m = weno.metrics(grid, times, sol['fields'], PHI_E, V_exact)
    weno.print_metrics(times, m, label=f'{name} — WENO5 + TVD-RK3')
    print(f"steps {sol['steps']} | {sol['minutes']:.1f} min | "
          f"peak GPU {sol['peak_MB']:.0f} MB")

    row = {'benchmark': name, 'scheme': 'WENO5+RK3', 'grid_n': GRID_N,
           'cfl': CFL, 'steps': sol['steps'],
           'minutes': f"{sol['minutes']:.3f}", 'peak_MB': f"{sol['peak_MB']:.0f}"}
    for k in ('l2_abs', 'rel_L2', 'vol_MAPE', 'eik_dev'):
        for i in range(N_SNAPSHOTS):
            row[f'{k}_t{i}'] = f'{m[k][i]:.6e}'
        row[f'avg_{k}'] = f"{m['avg_' + k]:.6e}"
    rows.append(row)

    np.savez_compressed(f'{OUTDIR}/{name}_weno_fields.npz',
                        **{f'f{i}': sol['fields'][t] for i, t in enumerate(times)})
    del sol, PHI_E
    if device == 'cuda':
        torch.cuda.empty_cache()

with open(CSV_PATH, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
print('\nwritten →', CSV_PATH)

## Comparison against the PINN

Paste the PINN numbers from each benchmark's `*_results.csv` into `PINN` below.
Both methods use the same grid, snapshots and error definitions, so the
comparison is direct.

In [ ]:
# --- side-by-side ------------------------------------------------------
PINN = {   # avg rel L2 (%), avg volume MAPE (%), wall-clock minutes
    'TR3D': (0.122, 1.143, 11.8),
    'RO3D': (0.160, 0.117, 15.6),
    'ZD3D': (0.480, 0.328, 24.9),
    'RV3D': (0.591, 0.564, 23.7),
}

print(f"{'benchmark':<10}{'rel L2 (%)':>22}{'volume MAPE (%)':>24}{'minutes':>18}")
print(f"{'':<10}{'WENO':>11}{'PINN':>11}{'WENO':>12}{'PINN':>12}{'WENO':>9}{'PINN':>9}")
print('-' * 74)
for r in rows:
    b = r['benchmark']; p = PINN.get(b, (None, None, None))
    f2 = lambda v: f'{v:.3f}' if v is not None else '--'
    print(f"{b:<10}{float(r['avg_rel_L2']):>11.3f}{f2(p[0]):>11}"
          f"{float(r['avg_vol_MAPE']):>12.3f}{f2(p[1]):>12}"
          f"{float(r['minutes']):>9.1f}{f2(p[2]):>9}")